In [0]:
from pyspark.sql import functions as F

gold_daily_demand = spark.table(
    "workspace.default.gold_daily_demand"
)

inventory_silver = spark.table(
    "workspace.default.silver_inventory"
)

print("Demand rows:", gold_daily_demand.count())
print("Inventory rows:", inventory_silver.count())

Demand rows: 16546616
Inventory rows: 284755


In [0]:
inventory_position = gold_daily_demand.alias("d").join(
    inventory_silver.alias("i"),
    (F.col("d.product_id") == F.col("i.product_id")) &
    (F.col("d.store") == F.col("i.store")) &
    (F.col("d.transaction_date") >= F.col("i.start_date")) &
    (
        F.col("d.transaction_date") <= F.col("i.end_date") |
        F.col("i.end_date").isNull()
    ),
    "left"
)

In [0]:
inventory_position = inventory_position.select(
    F.col("d.transaction_date"),
    F.col("d.product_id"),
    F.col("d.store"),

    F.col("d.daily_units_sold"),
    F.col("d.daily_sales"),
    F.col("d.daily_gross_margin"),
    F.col("d.daily_transactions"),
    F.col("d.return_count"),

    F.col("d.rolling_7d_units"),
    F.col("d.rolling_30d_units"),
    F.col("d.avg_daily_demand_7d"),
    F.col("d.avg_daily_demand_30d"),

    F.col("i.qty_on_hand"),
    F.col("i.stock_status"),
    F.col("i.store_type"),
    F.col("i.supplier"),
    F.col("i.unit_selling_price"),
    F.col("i.unit_cost_price"),
    F.col("i.is_current")
)

In [0]:
inventory_position = gold_daily_demand.alias("d").join(
    inventory_silver.alias("i"),
    (
        (F.col("d.product_id") == F.col("i.product_id")) &
        (F.col("d.store") == F.col("i.store")) &
        (F.col("d.transaction_date") >= F.col("i.start_date")) &
        (
            (F.col("d.transaction_date") <= F.col("i.end_date")) |
            F.col("i.end_date").isNull()
        )
    ),
    "left"
)

In [0]:
inventory_position = inventory_position.select(
    F.col("d.transaction_date"),
    F.col("d.product_id"),
    F.col("d.store"),

    F.col("d.daily_units_sold"),
    F.col("d.daily_sales"),
    F.col("d.daily_gross_margin"),
    F.col("d.daily_transactions"),
    F.col("d.return_count"),

    F.col("d.rolling_7d_units"),
    F.col("d.rolling_30d_units"),
    F.col("d.avg_daily_demand_7d"),
    F.col("d.avg_daily_demand_30d"),

    F.col("i.qty_on_hand"),
    F.col("i.stock_status"),
    F.col("i.store_type"),
    F.col("i.supplier"),
    F.col("i.unit_selling_price"),
    F.col("i.unit_cost_price"),
    F.col("i.is_current")
)

In [0]:
inventory_position.select(
    "transaction_date",
    "product_id",
    "store",
    "daily_units_sold",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "stock_status",
    "is_current"
).orderBy(
    F.desc("transaction_date")
).show(20, False)

+----------------+-----------+--------+----------------+--------------------+-----------+------------+----------+
|transaction_date|product_id |store   |daily_units_sold|avg_daily_demand_30d|qty_on_hand|stock_status|is_current|
+----------------+-----------+--------+----------------+--------------------+-----------+------------+----------+
|2026-04-24      |PROD-100044|STR-1010|0.0             |0.0                 |3          |Full Price  |true      |
|2026-04-24      |PROD-100043|STR-1348|0.0             |0.0                 |1          |Full Price  |true      |
|2026-04-24      |PROD-100043|STR-1010|0.0             |0.0                 |2          |Full Price  |true      |
|2026-04-24      |PROD-100043|STR-1203|0.0             |0.0                 |2          |Full Price  |true      |
|2026-04-24      |PROD-100043|STR-1305|0.0             |0.0                 |1          |Full Price  |true      |
|2026-04-24      |PROD-100043|STR-1074|0.0             |0.0                 |1          

In [0]:
print("Inventory Position rows:", inventory_position.count())

print(
    "Rows with no inventory match:",
    inventory_position.filter(
        F.col("qty_on_hand").isNull()
    ).count()
)

Inventory Position rows: 16546616
Rows with no inventory match: 9281301


In [0]:
inventory_position.withColumn(
    "inventory_match",
    F.when(
        F.col("qty_on_hand").isNull(),
        "Missing"
    ).otherwise("Matched")
).groupBy(
    "inventory_match"
).agg(
    F.count("*").alias("rows")
).show()

+---------------+-------+
|inventory_match|   rows|
+---------------+-------+
|        Matched|7265315|
|        Missing|9281301|
+---------------+-------+



In [0]:
inventory_position.filter(
    F.col("qty_on_hand").isNull()
).groupBy(
    "transaction_date"
).count().orderBy(
    "transaction_date"
).show(30, False)

+----------------+-----+
|transaction_date|count|
+----------------+-----+
|2025-06-01      |48858|
|2025-06-02      |47937|
|2025-06-03      |47384|
|2025-06-04      |47118|
|2025-06-05      |46901|
|2025-06-06      |46710|
|2025-06-07      |46474|
|2025-06-08      |46103|
|2025-06-09      |45822|
|2025-06-10      |45462|
|2025-06-11      |44997|
|2025-06-12      |44228|
|2025-06-13      |43320|
|2025-06-14      |43134|
|2025-06-15      |43016|
|2025-06-16      |42748|
|2025-06-17      |42501|
|2025-06-18      |42006|
|2025-06-19      |41909|
|2025-06-20      |41629|
|2025-06-21      |41475|
|2025-06-22      |41346|
|2025-06-23      |41290|
|2025-06-24      |41238|
|2025-06-25      |40924|
|2025-06-26      |40796|
|2025-06-27      |40654|
|2025-06-28      |40509|
|2025-06-29      |40139|
|2025-06-30      |40086|
+----------------+-----+
only showing top 30 rows


In [0]:
inventory_position.filter(
    F.col("qty_on_hand").isNull()
).groupBy(
    "product_id",
    "store"
).count().orderBy(
    F.desc("count")
).show(20, False)

+-----------+--------+-----+
|product_id |store   |count|
+-----------+--------+-----+
|PROD-145916|STR-1194|328  |
|PROD-146039|STR-1107|328  |
|PROD-144469|STR-1149|328  |
|PROD-141933|STR-1353|328  |
|PROD-139487|STR-1198|328  |
|PROD-145637|STR-1149|328  |
|PROD-145810|STR-1010|328  |
|PROD-145803|STR-1094|328  |
|PROD-152049|STR-1148|328  |
|PROD-152050|STR-1353|328  |
|PROD-151093|STR-1348|328  |
|PROD-147348|STR-1344|328  |
|PROD-146006|STR-1082|328  |
|PROD-145730|STR-1148|328  |
|PROD-152159|STR-1194|328  |
|PROD-146000|STR-1237|328  |
|PROD-152159|STR-1157|328  |
|PROD-122003|STR-1157|328  |
|PROD-152064|STR-1107|328  |
|PROD-152064|STR-1335|328  |
+-----------+--------+-----+
only showing top 20 rows


In [0]:
inventory_gap_analysis = inventory_position.groupBy(
    "product_id",
    "store"
).agg(
    F.count("*").alias("total_days"),
    F.sum(
        F.when(
            F.col("qty_on_hand").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_inventory_days"),
    F.min(
        F.when(
            F.col("qty_on_hand").isNotNull(),
            F.col("transaction_date")
        )
    ).alias("first_inventory_date"),
    F.max(
        F.when(
            F.col("qty_on_hand").isNotNull(),
            F.col("transaction_date")
        )
    ).alias("last_inventory_date")
).withColumn(
    "inventory_coverage_status",
    F.when(
        F.col("missing_inventory_days") == F.col("total_days"),
        "No Inventory History"
    ).when(
        F.col("missing_inventory_days") > 0,
        "Partial Inventory History"
    ).otherwise(
        "Complete Inventory History"
    )
)

inventory_gap_analysis.groupBy(
    "inventory_coverage_status"
).agg(
    F.count("*").alias("product_store_pairs"),
    F.sum("missing_inventory_days").alias("missing_days")
).show()

+-------------------------+-------------------+------------+
|inventory_coverage_status|product_store_pairs|missing_days|
+-------------------------+-------------------+------------+
|     Partial Inventory...|              47482|     8432437|
|     Complete Inventor...|                377|           0|
|     No Inventory History|               2588|      848864|
+-------------------------+-------------------+------------+



In [0]:
inventory_gap_analysis.orderBy(
    F.desc("missing_inventory_days")
).show(30, False)

+-----------+--------+----------+----------------------+--------------------+-------------------+-------------------------+
|product_id |store   |total_days|missing_inventory_days|first_inventory_date|last_inventory_date|inventory_coverage_status|
+-----------+--------+----------+----------------------+--------------------+-------------------+-------------------------+
|PROD-152159|STR-1157|328       |328                   |NULL                |NULL               |No Inventory History     |
|PROD-152392|STR-1107|328       |328                   |NULL                |NULL               |No Inventory History     |
|PROD-144469|STR-1149|328       |328                   |NULL                |NULL               |No Inventory History     |
|PROD-145810|STR-1010|328       |328                   |NULL                |NULL               |No Inventory History     |
|PROD-145730|STR-1148|328       |328                   |NULL                |NULL               |No Inventory History     |
|PROD-14

In [0]:
partial_inventory = inventory_gap_analysis.filter(
    F.col("inventory_coverage_status") == "Partial Inventory History"
)

partial_inventory.select(
    "product_id",
    "store",
    "total_days",
    "missing_inventory_days",
    "first_inventory_date",
    "last_inventory_date"
).withColumn(
    "dataset_start_date",
    F.lit("2025-06-01").cast("date")
).withColumn(
    "dataset_end_date",
    F.lit("2026-04-24").cast("date")
).withColumn(
    "expected_pre_inventory_days",
    F.datediff(
        F.col("first_inventory_date"),
        F.col("dataset_start_date")
    )
).withColumn(
    "expected_post_inventory_days",
    F.datediff(
        F.col("dataset_end_date"),
        F.col("last_inventory_date")
    )
).withColumn(
    "unexplained_missing_days",
    F.col("missing_inventory_days")
    - F.col("expected_pre_inventory_days")
    - F.col("expected_post_inventory_days")
).select(
    "expected_pre_inventory_days",
    "expected_post_inventory_days",
    "unexplained_missing_days"
).agg(
    F.sum(
        F.when(
            F.col("unexplained_missing_days") > 0,
            1
        ).otherwise(0)
    ).alias("pairs_with_unexplained_gaps"),
    F.sum("unexplained_missing_days").alias("total_unexplained_days")
).show()

+---------------------------+----------------------+
|pairs_with_unexplained_gaps|total_unexplained_days|
+---------------------------+----------------------+
|                       6926|                291820|
+---------------------------+----------------------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

inventory_history = inventory_silver.select(
    "product_id",
    "store",
    "start_date",
    "end_date"
).withColumn(
    "previous_end_date",
    F.lag("end_date").over(
        Window.partitionBy(
            "product_id",
            "store"
        ).orderBy("start_date")
    )
)

inventory_gaps = inventory_history.filter(
    F.col("previous_end_date").isNotNull()
).withColumn(
    "gap_days",
    F.datediff(
        F.col("start_date"),
        F.col("previous_end_date")
    ) - 1
).filter(
    F.col("gap_days") > 0
)

print("Internal inventory gaps:", inventory_gaps.count())

inventory_gaps.select(
    "product_id",
    "store",
    "previous_end_date",
    "start_date",
    "gap_days"
).orderBy(
    F.desc("gap_days")
).show(30, False)

Internal inventory gaps: 8607
+-----------+--------+-----------------+----------+--------+
|product_id |store   |previous_end_date|start_date|gap_days|
+-----------+--------+-----------------+----------+--------+
|PROD-160856|STR-1080|2025-06-20       |2026-04-15|298     |
|PROD-165960|STR-1338|2025-06-04       |2026-03-20|288     |
|PROD-161781|STR-1107|2025-06-12       |2026-03-23|283     |
|PROD-129498|STR-1237|2025-06-12       |2026-03-13|273     |
|PROD-158658|STR-1198|2025-06-24       |2026-03-18|266     |
|PROD-160853|STR-1109|2025-07-17       |2026-04-07|263     |
|PROD-160854|STR-1114|2025-06-29       |2026-03-13|256     |
|PROD-170508|STR-1235|2025-07-29       |2026-04-12|256     |
|PROD-170510|STR-1255|2025-08-06       |2026-04-14|250     |
|PROD-169349|STR-1198|2025-06-04       |2026-02-09|249     |
|PROD-129492|STR-1044|2025-07-10       |2026-03-13|245     |
|PROD-165154|STR-1198|2025-07-27       |2026-03-29|244     |
|PROD-165149|STR-1198|2025-08-01       |2026-03-29|239 

In [0]:
inventory_gaps.agg(
    F.count("*").alias("gap_periods"),
    F.sum("gap_days").alias("total_gap_days"),
    F.max("gap_days").alias("largest_gap_days"),
    F.avg("gap_days").alias("average_gap_days")
).show()

+-----------+--------------+----------------+-----------------+
|gap_periods|total_gap_days|largest_gap_days| average_gap_days|
+-----------+--------------+----------------+-----------------+
|       8607|        299671|             298|34.81712559544557|
+-----------+--------------+----------------+-----------------+



In [0]:
no_inventory_history = inventory_gap_analysis.filter(
    F.col("inventory_coverage_status") == "No Inventory History"
).select(
    "product_id",
    "store"
).withColumn(
    "no_inventory_history",
    F.lit(True)
)

inventory_position = inventory_position.join(
    no_inventory_history,
    ["product_id", "store"],
    "left"
).withColumn(
    "inventory_data_status",
    F.when(
        F.col("qty_on_hand").isNotNull(),
        "Inventory Available"
    ).when(
        F.col("no_inventory_history") == True,
        "No Inventory History"
    ).otherwise(
        "Inventory Gap"
    )
).drop(
    "no_inventory_history"
)

In [0]:
inventory_position.groupBy(
    "inventory_data_status"
).count().show()

+---------------------+-------+
|inventory_data_status|  count|
+---------------------+-------+
| No Inventory History| 848864|
|  Inventory Available|7265315|
|        Inventory Gap|8432437|
+---------------------+-------+



In [0]:
from pyspark.sql import functions as F

inventory_position = inventory_position.withColumn(
    "inventory_coverage_days",
    F.when(
        (F.col("qty_on_hand").isNotNull()) &
        (F.col("avg_daily_demand_30d") > 0),
        F.col("qty_on_hand") / F.col("avg_daily_demand_30d")
    ).otherwise(F.lit(None).cast("double"))
)

inventory_position.select(
    "transaction_date",
    "product_id",
    "store",
    "daily_units_sold",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "inventory_coverage_days",
    "inventory_data_status"
).filter(
    F.col("inventory_data_status") == "Inventory Available"
).orderBy(
    F.col("inventory_coverage_days").asc_nulls_last()
).show(30, False)

+----------------+-----------+--------+----------------+--------------------+-----------+-----------------------+---------------------+
|transaction_date|product_id |store   |daily_units_sold|avg_daily_demand_30d|qty_on_hand|inventory_coverage_days|inventory_data_status|
+----------------+-----------+--------+----------------+--------------------+-----------+-----------------------+---------------------+
|2026-02-24      |PROD-100439|STR-1075|0.0             |0.03333333333333333 |-9         |-270.0                 |Inventory Available  |
|2026-02-27      |PROD-100439|STR-1075|0.0             |0.03333333333333333 |-9         |-270.0                 |Inventory Available  |
|2026-02-25      |PROD-100439|STR-1075|0.0             |0.03333333333333333 |-9         |-270.0                 |Inventory Available  |
|2026-02-26      |PROD-100439|STR-1075|0.0             |0.03333333333333333 |-9         |-270.0                 |Inventory Available  |
|2026-03-01      |PROD-100437|STR-1082|0.0      

In [0]:
from pyspark.sql import functions as F

inventory_position.filter(
    F.col("inventory_data_status") == "Inventory Available"
).groupBy(
    F.when(F.col("qty_on_hand") < 0, "Negative Stock")
     .when(F.col("qty_on_hand") == 0, "Zero Stock")
     .otherwise("Positive Stock")
     .alias("stock_position")
).agg(
    F.count("*").alias("rows")
).orderBy(
    "stock_position"
).show()

+--------------+-------+
|stock_position|   rows|
+--------------+-------+
|Negative Stock|  46571|
|Positive Stock|6484450|
|    Zero Stock| 734294|
+--------------+-------+



In [0]:
from pyspark.sql import functions as F

inventory_position = inventory_position.withColumn(
    "inventory_risk",
    F.when(
        F.col("inventory_data_status") == "No Inventory History",
        "Unknown - No Inventory History"
    ).when(
        F.col("inventory_data_status") == "Inventory Gap",
        "Unknown - Inventory Gap"
    ).when(
        F.col("qty_on_hand") < 0,
        "Inventory Deficit"
    ).when(
        (F.col("qty_on_hand") == 0) &
        (F.col("avg_daily_demand_30d") > 0),
        "Stockout Risk"
    ).when(
        (F.col("qty_on_hand") == 0) &
        (F.col("avg_daily_demand_30d") == 0),
        "Zero Stock - No Recent Demand"
    ).when(
        (F.col("inventory_coverage_days") <= 7) &
        (F.col("avg_daily_demand_30d") > 0),
        "Critical - <=7 Days"
    ).when(
        (F.col("inventory_coverage_days") <= 14) &
        (F.col("avg_daily_demand_30d") > 0),
        "High Risk - 8-14 Days"
    ).when(
        (F.col("inventory_coverage_days") <= 30) &
        (F.col("avg_daily_demand_30d") > 0),
        "Medium Risk - 15-30 Days"
    ).when(
        (F.col("inventory_coverage_days") > 90) &
        (F.col("avg_daily_demand_30d") > 0),
        "Overstock Risk - >90 Days"
    ).when(
        (F.col("avg_daily_demand_30d") == 0) &
        (F.col("qty_on_hand") > 0),
        "No Recent Demand"
    ).otherwise(
        "Healthy"
    )
)

In [0]:
inventory_position.groupBy(
    "inventory_risk"
).count().orderBy(
    F.desc("count")
).show(False)

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-5486175038803850>, line 5
      1 inventory_position.groupBy(
      2     "inventory_risk"
      3 ).count().orderBy(
      4     F.desc("count")
----> 5 ).show(False)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1297, in DataFrame.show(self, n, truncate, vertical)
   1296 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1297     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:899, in DataFrame._show_string(self, n, truncate, vertical)
    895 def _show_string(
    896     self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False
    897 ) -> str:
    898     if not isinstance(n, int) or isinstance(n, bool):
--> 899         rais

In [0]:
inventory_position.groupBy(
    "inventory_risk"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+------------------------------+-------+
|inventory_risk                |count  |
+------------------------------+-------+
|Unknown - Inventory Gap       |8432437|
|No Recent Demand              |5016018|
|Unknown - No Inventory History|848864 |
|Medium Risk - 15-30 Days      |754130 |
|Healthy                       |529443 |
|Zero Stock - No Recent Demand |493042 |
|Stockout Risk                 |240543 |
|Overstock Risk - >90 Days     |154522 |
|Inventory Deficit             |46571  |
|High Risk - 8-14 Days         |27332  |
|Critical - <=7 Days           |3714   |
+------------------------------+-------+



In [0]:
from pyspark.sql import functions as F

latest_date = inventory_position.agg(
    F.max("transaction_date").alias("latest_date")
).collect()[0]["latest_date"]

print("Latest business date:", latest_date)

current_inventory = inventory_position.filter(
    F.col("transaction_date") == F.lit(latest_date)
)

print("Current snapshot rows:", current_inventory.count())

current_inventory.select(
    "transaction_date",
    "product_id",
    "store",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "inventory_coverage_days",
    "inventory_data_status",
    "inventory_risk"
).show(20, truncate=False)

Latest business date: 2026-04-24
Current snapshot rows: 50447
+----------------+-----------+--------+--------------------+-----------+-----------------------+---------------------+-----------------------+
|transaction_date|product_id |store   |avg_daily_demand_30d|qty_on_hand|inventory_coverage_days|inventory_data_status|inventory_risk         |
+----------------+-----------+--------+--------------------+-----------+-----------------------+---------------------+-----------------------+
|2026-04-24      |PROD-100043|STR-1006|0.0                 |1          |NULL                   |Inventory Available  |No Recent Demand       |
|2026-04-24      |PROD-100043|STR-1010|0.0                 |2          |NULL                   |Inventory Available  |No Recent Demand       |
|2026-04-24      |PROD-100043|STR-1026|0.0                 |1          |NULL                   |Inventory Available  |No Recent Demand       |
|2026-04-24      |PROD-100043|STR-1074|0.0                 |1          |NULL    

In [0]:
from pyspark.sql import functions as F

current_inventory = current_inventory.withColumn(
    "target_stock_14d",
    F.when(
        (F.col("inventory_data_status") == "Inventory Available") &
        (F.col("avg_daily_demand_30d") > 0),
        F.ceil(F.col("avg_daily_demand_30d") * F.lit(14))
    ).otherwise(
        F.lit(None).cast("long")
    )
)

current_inventory = current_inventory.withColumn(
    "recommended_order_qty",
    F.when(
        F.col("target_stock_14d").isNotNull() &
        F.col("qty_on_hand").isNotNull(),
        F.greatest(
            F.col("target_stock_14d") - F.col("qty_on_hand"),
            F.lit(0)
        )
    ).otherwise(
        F.lit(None).cast("long")
    )
)

current_inventory.select(
    "product_id",
    "store",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "inventory_coverage_days",
    "inventory_risk",
    "target_stock_14d",
    "recommended_order_qty"
).filter(
    F.col("recommended_order_qty").isNotNull()
).orderBy(
    F.desc("recommended_order_qty")
).show(30, truncate=False)

+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|product_id |store   |avg_daily_demand_30d|qty_on_hand|inventory_coverage_days|inventory_risk     |target_stock_14d|recommended_order_qty|
+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|PROD-100804|STR-1235|0.4                 |0          |0.0                    |Stockout Risk      |6               |6                    |
|PROD-100805|STR-1080|0.36666666666666664 |1          |2.7272727272727275     |Critical - <=7 Days|6               |5                    |
|PROD-100803|STR-1080|0.3333333333333333  |0          |0.0                    |Stockout Risk      |5               |5                    |
|PROD-100805|STR-1338|0.4                 |1          |2.5                    |Critical - <=7 Days|6               |5                    |
|PROD-100803|STR-1235|0.4  

In [0]:
replenishment_recommendations = current_inventory.filter(
    (F.col("inventory_data_status") == "Inventory Available") &
    (F.col("avg_daily_demand_30d") > 0)
).select(
    "transaction_date",
    "product_id",
    "store",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "inventory_coverage_days",
    "inventory_risk",
    "target_stock_14d",
    "recommended_order_qty"
)

print("Recommendation rows:", replenishment_recommendations.count())

Recommendation rows: 3763


In [0]:
replenishment_recommendations.orderBy(
    F.desc("recommended_order_qty")
).show(30, truncate=False)

+----------------+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|transaction_date|product_id |store   |avg_daily_demand_30d|qty_on_hand|inventory_coverage_days|inventory_risk     |target_stock_14d|recommended_order_qty|
+----------------+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|2026-04-24      |PROD-100804|STR-1235|0.4                 |0          |0.0                    |Stockout Risk      |6               |6                    |
|2026-04-24      |PROD-100805|STR-1080|0.36666666666666664 |1          |2.7272727272727275     |Critical - <=7 Days|6               |5                    |
|2026-04-24      |PROD-100803|STR-1080|0.3333333333333333  |0          |0.0                    |Stockout Risk      |5               |5                    |
|2026-04-24      |PROD-100805|STR-1338|0.4                 |1   

In [0]:
replenishment_recommendations.groupBy(
    "inventory_risk"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+-------------------------+-----+
|inventory_risk           |count|
+-------------------------+-----+
|Medium Risk - 15-30 Days |1546 |
|Healthy                  |989  |
|Stockout Risk            |752  |
|Overstock Risk - >90 Days|356  |
|Inventory Deficit        |58   |
|High Risk - 8-14 Days    |49   |
|Critical - <=7 Days      |13   |
+-------------------------+-----+



In [0]:
replenishment_recommendations.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_replenishment_recommendations")

print("Gold Replenishment Recommendations table created successfully.")

Gold Replenishment Recommendations table created successfully.


In [0]:
print(
    "Rows:",
    spark.table("workspace.default.gold_replenishment_recommendations").count()
)

Rows: 3763


In [0]:
spark.table(
    "workspace.default.gold_replenishment_recommendations"
).orderBy(
    F.desc("recommended_order_qty"),
    F.asc("inventory_coverage_days")
).show(30, truncate=False)

+----------------+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|transaction_date|product_id |store   |avg_daily_demand_30d|qty_on_hand|inventory_coverage_days|inventory_risk     |target_stock_14d|recommended_order_qty|
+----------------+-----------+--------+--------------------+-----------+-----------------------+-------------------+----------------+---------------------+
|2026-04-24      |PROD-100804|STR-1235|0.4                 |0          |0.0                    |Stockout Risk      |6               |6                    |
|2026-04-24      |PROD-100803|STR-1080|0.3333333333333333  |0          |0.0                    |Stockout Risk      |5               |5                    |
|2026-04-24      |PROD-100805|STR-1338|0.4                 |1          |2.5                    |Critical - <=7 Days|6               |5                    |
|2026-04-24      |PROD-100803|STR-1235|0.4                 |1   

In [0]:
gold_inventory_risk = current_inventory.select(
    "transaction_date",
    "product_id",
    "store",
    "avg_daily_demand_30d",
    "qty_on_hand",
    "inventory_coverage_days",
    "inventory_data_status",
    "inventory_risk",
    "target_stock_14d",
    "recommended_order_qty"
)

print("Gold Inventory Risk rows:", gold_inventory_risk.count())

Gold Inventory Risk rows: 50447


In [0]:
gold_inventory_risk.groupBy(
    "inventory_risk"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

+------------------------------+-----+
|inventory_risk                |count|
+------------------------------+-----+
|Unknown - Inventory Gap       |33172|
|No Recent Demand              |8803 |
|Unknown - No Inventory History|2588 |
|Zero Stock - No Recent Demand |1845 |
|Medium Risk - 15-30 Days      |1546 |
|Healthy                       |1088 |
|Stockout Risk                 |752  |
|Overstock Risk - >90 Days     |356  |
|Inventory Deficit             |235  |
|High Risk - 8-14 Days         |49   |
|Critical - <=7 Days           |13   |
+------------------------------+-----+



In [0]:
gold_inventory_risk.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_inventory_risk")

print("Gold Inventory Risk table created successfully.")

Gold Inventory Risk table created successfully.


In [0]:
print(
    "Gold Inventory Risk:",
    spark.table("workspace.default.gold_inventory_risk").count()
)

print(
    "Gold Replenishment Recommendations:",
    spark.table("workspace.default.gold_replenishment_recommendations").count()
)

Gold Inventory Risk: 50447
Gold Replenishment Recommendations: 3763


In [0]:
print("Gold Daily Demand:",
      spark.table("workspace.default.gold_daily_demand").count())

print("Gold Inventory Risk:",
      spark.table("workspace.default.gold_inventory_risk").count())

print("Gold Replenishment Recommendations:",
      spark.table("workspace.default.gold_replenishment_recommendations").count())

Gold Daily Demand: 16546616
Gold Inventory Risk: 50447
Gold Replenishment Recommendations: 3763
